In [31]:
from pathlib import Path
import pandas as pd

RAW = Path("../../data/raw")
competitions = pd.read_csv(RAW/"competitions.csv")
clubs = pd.read_csv(RAW/"clubs.csv")
players = pd.read_csv(RAW/"players.csv", parse_dates=["date_of_birth", "contract_expiration_date"])
valuations = pd.read_csv(RAW/"player_valuations.csv", parse_dates=["date"])
transfers = pd.read_csv(RAW/"transfers.csv", parse_dates=["transfer_date"])
salaries = pd.read_csv(RAW/"american_soccer_analysis_mls_salaries_players.csv")

for n, df in [("competition", competitions), ("clubs", clubs), ("players", players), ("valuations", valuations), ("transfers", transfers), ("salaries", salaries)]:
    print(n, df.shape)

competition (65, 11)
clubs (796, 17)
players (50149, 26)
valuations (656301, 6)
transfers (175165, 10)
salaries (15490, 8)


In [32]:
# MLS clubs
print(competitions[competitions.competition_id == "MLS1"].to_string())
mls_clubs = clubs[clubs.domestic_competition_id == "MLS1"]
print(len(mls_clubs))
print(mls_clubs[["club_id", "name", "last_season"]].sort_values("name").to_string())

   competition_id     competition_code                 name    sub_type             type  country_id   country_name domestic_league_code confederation  total_clubs                                                                             url
38           MLS1  major-league-soccer  major-league-soccer  first_tier  domestic_league         184  United States                 MLS1       amerika         30.0  https://www.transfermarkt.co.uk/major-league-soccer/startseite/wettbewerb/MLS1
30
     club_id                                name  last_season
556    51663        Atlanta United Football Club         2025
691    72309                           Austin FC         2025
708    78435             Charlotte Football Club         2025
507      432            Chicago Fire Soccer Club         2025
665    69261  Club Internacional de Fútbol Miami         2025
479     4078               Club de Foot Montréal         2025
104     1247                     Colorado Rapids         2025
719      813 

In [33]:
# Valuations: MLS rows and unique players per year
mv = valuations[valuations.player_club_domestic_competition_id == "MLS1"]
print(len(mv))
print(mv.groupby(mv.date.dt.year).agg(rows=("player_id", "size"), players=("player_id", "nunique")))

9351
      rows  players
date               
2004     1        1
2006     1        1
2007     1        1
2008     8        5
2009    17        8
2010    16        7
2011    19       12
2012    26       16
2013    28       17
2014    28       15
2015    70       54
2016   217       91
2017   264      136
2018   383      180
2019   460      225
2020   924      330
2021   833      399
2022   939      457
2023  1150      555
2024  1312      622
2025  1686      745
2026   968      806


In [34]:
# Transfers touching MLS clubs, 2016-2025
ids = set(mls_clubs.club_id)
tm = transfers[(transfers.from_club_id.isin(ids) | transfers.to_club_id.isin(ids))
        & transfers.transfer_date.between("2016-01-01", "2025-12-31")]
print(len(tm))
print(tm.groupby(tm.transfer_date.dt.year).size())
print(tm.transfer_fee.isna().sum(), (tm.transfer_fee == 0).sum())
print(tm[["from_club_name", "to_club_name", "transfer_fee"]].head(15).to_string())

3151
transfer_date
2016     98
2017    157
2018    182
2019    281
2020    237
2021    320
2022    379
2023    384
2024    493
2025    620
dtype: int64
581 1857
       from_club_name   to_club_name  transfer_fee
8301          Toronto  Vasco da Gama           0.0
8309            Miami       Atlético           0.0
8313        Charlotte     Olympiacos           0.0
8325        San Diego  Celta de Vigo           0.0
8329        Vancouver         Chivas           0.0
8355        San Diego          Pafos           0.0
8356       Cincinnati        Udinese           0.0
8377          Chicago          Porto           0.0
8378            Atlas       Montréal           0.0
8395  Louisville City      Salt Lake           0.0
8397    Górnik Zabrze      Minnesota           0.0
8401            Miami   Boca Juniors           0.0
8407   St. Louis CITY   Slavia Praha           0.0
8409       Cincinnati  Middlesbrough           0.0
8412      New England      FC Nantes           0.0


In [35]:
# ASA salaries
print(salaries.Season.value_counts().sort_index())
print(salaries.Team.nunique(), salaries.Player.nunique())
print(salaries.head(10).to_string())

Season
2016    1128
2017    1267
2018    1345
2019    1386
2020     771
2021    1636
2022    1675
2023    1784
2024    1761
2025    1826
2026     911
Name: count, dtype: int64
30 2829
   Unnamed: 0           Player Team  Season Position  Base Salary Guaranteed Compensation        Date
0         NaN     Lionel Messi  MIA    2026        F  $25,000,000             $28,333,332  2026-04-16
1         NaN     Lionel Messi  MIA    2023        M  $12,000,000             $20,446,668  2023-09-15
2         NaN     Lionel Messi  MIA    2024        F  $12,000,000             $20,446,668  2024-04-25
3         NaN     Lionel Messi  MIA    2024        F  $12,000,000             $20,446,668  2024-09-13
4         NaN     Lionel Messi  MIA    2025        F  $12,000,000             $20,446,668  2025-05-23
5         NaN     Lionel Messi  MIA    2025        F  $12,000,000             $20,446,668  2025-10-01
6         NaN  Lorenzo Insigne  TOR    2023        M   $7,500,000             $15,400,000  2023-09-15


In [36]:
import numpy as np

# A. ASA structure
salaries = salaries.drop(columns=["Unnamed: 0"])
print(salaries.groupby(["Season", "Date"]).size().to_string())
for c in ["Base Salary", "Guaranteed Compensation"]:
    salaries[c + "_num"] = pd.to_numeric(salaries[c].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
    bad = salaries[salaries[c + "_num"].isna()][c]
    print(c, "unparsed:", len(bad), bad.unique()[:10])
g = salaries.groupby(["Player", "Season"])
print("player-seasons:", g.ngroups)
print("with >1 release:", (g.size() > 1).sum())
print("with >1 distinct guaranteed value:", (g["Guaranteed Compensation_num"].nunique() > 1).sum())
print(sorted(salaries.Team.unique()))
print(salaries.Position.value_counts())

Season  Date      
2016    2016-05-15    548
        2016-09-15    580
2017    2017-04-15    614
        2017-09-15    653
2018    2018-05-01    661
        2018-09-15    684
2019    2019-06-01    681
        2019-09-13    705
2020    2020-12-01    771
2021    2021-04-15    781
        2021-09-30    855
2022    2022-04-15    829
        2022-09-02    846
2023    2023-04-30    869
        2023-09-15    915
2024    2024-04-25    867
        2024-09-13    894
2025    2025-05-23    893
        2025-10-01    933
2026    2026-04-16    911
Base Salary unparsed: 0 <StringArray>
[]
Length: 0, dtype: str
Guaranteed Compensation unparsed: 0 <StringArray>
[]
Length: 0, dtype: str
player-seasons: 9107
with >1 release: 6378
with >1 distinct guaranteed value: 345
['ATL', 'ATX', 'CHI', 'CIN', 'CLB', 'CLT', 'COL', 'DCU', 'FCD', 'HOU', 'LAFC', 'LAG', 'MIA', 'MIN', 'MTL', 'NER', 'NSH', 'NYC', 'NYRB', 'ORL', 'PHI', 'POR', 'RSL', 'SD', 'SEA', 'SJE', 'SKC', 'STL', 'TOR', 'VAN']
Position
D     5332
M     504

In [37]:
# Coverage: ASA players vs valued players, by year

asa_n = salaries.groupby("Season").Player.nunique().rename("asa_players")
mv_n = mv.groupby(mv.date.dt.year).player_id.nunique().rename("tm_valued_players")
cov = pd.concat([asa_n, mv_n], axis=1).loc[2016:2025]
cov["ratio"] = cov.tm_valued_players / cov.asa_players
print(cov)

      asa_players  tm_valued_players     ratio
2016        605.0                 91  0.150413
2017        676.0                136  0.201183
2018        713.0                180  0.252454
2019        732.0                225  0.307377
2020        771.0                330  0.428016
2021        881.0                399  0.452894
2022        919.0                457  0.497280
2023        957.0                555  0.579937
2024        954.0                622  0.651992
2025        988.0                745  0.754049


In [38]:
# Is the club column as-of-date? Look at players with MLS transfers

ids = tm.player_id.drop_duplicates().head(6)
print(valuations[valuations.player_id.isin(ids)].sort_values(["player_id", "date"])
      [["player_id", "date", "market_value_in_eur", "current_club_name", "player_club_domestic_competition_id"]]
      .to_string())

        player_id       date  market_value_in_eur               current_club_name player_club_domestic_competition_id
115503     207561 2014-02-14                25000                     Servette FC                                  C1
127527     207561 2014-09-12               250000                     Servette FC                                  C1
131156     207561 2014-12-15               300000                     Servette FC                                  C1
161579     207561 2015-12-18               300000                   FC Zürich U21                                 NaN
183657     207561 2016-07-16               400000                       FC Zürich                                  C1
195432     207561 2016-12-20               400000               FC Lausanne-Sport                                  C1
221876     207561 2017-07-12               400000               FC Lausanne-Sport                                  C1
235783     207561 2018-01-08               400000       

In [39]:
# Transfer fees
print((tm.transfer_fee > 0).sum())
print(tm.sort_values("transfer_fee", ascending=False)
      [["transfer_date", "player_name", "from_club_name", "to_club_name", "transfer_fee", "market_value_in_eur"]]
      .head(20).to_string())
kind = np.select([tm.transfer_fee.isna(), tm.transfer_fee == 0], ["null", "zero"], "positive")
print(tm.groupby(kind).market_value_in_eur.agg(rows="size", with_mv="count"))
print("duplicate player+date:", tm.duplicated(["player_id", "transfer_date"]).sum())

713
       transfer_date          player_name   from_club_name   to_club_name  transfer_fee  market_value_in_eur
55733     2023-01-23           Jhon Durán          Chicago    Aston Villa    29500000.0            6000000.0
29637     2024-07-10        Thiago Almada          Atlanta       Botafogo    24150000.0           27000000.0
111767    2019-01-31       Miguel Almirón          Atlanta      Newcastle    24000000.0           15000000.0
11750     2025-08-06        Heung-min Son        Tottenham           LAFC    22000000.0           20000000.0
20597     2025-02-04  Emmanuel Latte Lath    Middlesbrough        Atlanta    21250000.0           10000000.0
46221     2023-07-21     Taty Castellanos    New York City          Lazio    18600000.0           15000000.0
73143     2022-01-03         Ricardo Pepi           Dallas       Augsburg    16360000.0            8000000.0
90849     2020-09-07     Gonzalo Martínez          Atlanta       Al-Nassr    16000000.0           12000000.0
43936     2023-

In [40]:
# Name-level coverage diagnostic

import unicodedata, re

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z\s]", " ", s)).strip()

players["name_n"] = players["name"].map(norm)
salaries["name_n"] = salaries["Player"].map(norm)

mls_val_names = set(players.loc[players.player_id.isin(mv.player_id), "name_n"])
ps = (salaries.groupby(["Season", "name_n"], as_index=False)
         .agg(guaranteed=("Guaranteed Compensation_num", "max"), pos=("Position", "first")))
ps["in_players"]  = ps.name_n.isin(players.name_n)
ps["has_mls_val"] = ps.name_n.isin(mls_val_names)

print(ps.groupby("Season")[["in_players", "has_mls_val"]].mean().round(2))
ps["band"] = pd.cut(ps.guaranteed, [0, 1e5, 2e5, 5e5, 1e6, 2e6, np.inf], right=False)
print(ps.groupby("band", observed=True)[["in_players", "has_mls_val"]].agg(["mean", "size"]).round(2))
print(ps.groupby("pos")[["in_players", "has_mls_val"]].mean().round(2))
print("names with >1 players.csv match:", (players.name_n.value_counts() > 1).sum())

        in_players  has_mls_val
Season                         
2016          0.33         0.12
2017          0.40         0.18
2018          0.47         0.24
2019          0.54         0.32
2020          0.60         0.40
2021          0.63         0.44
2022          0.68         0.51
2023          0.73         0.57
2024          0.82         0.67
2025          0.91         0.77
2026          0.92         0.85
                       in_players       has_mls_val      
                             mean  size        mean  size
band                                                     
[0.0, 100000.0)              0.46  2633        0.34  2633
[100000.0, 200000.0)         0.61  1747        0.48  1747
[200000.0, 500000.0)         0.73  2125        0.53  2125
[500000.0, 1000000.0)        0.84  1632        0.62  1632
[1000000.0, 2000000.0)       0.88   603        0.66   603
[2000000.0, inf)             0.93   362        0.67   362
     in_players  has_mls_val
pos                         
D   

In [43]:
# Final coverage check: ASA player-season-club vs. last MLS-club valuation within 365 days before the ASA snapshot

xwalk = {"ATL":51663,"ATX":72309,"CHI":432,"CIN":51772,"CLB":813,"CLT":78435,"COL":1247,
         "DCU":2440,"FCD":8816,"HOU":9168,"LAFC":51828,"LAG":1061,"MIA":69261,"MIN":56089,
         "MTL":4078,"NER":626,"NSH":63966,"NYC":40058,"NYRB":623,"ORL":45604,"PHI":25467,
         "POR":4291,"RSL":6643,"SD":114977,"SEA":9636,"SJE":218,"SKC":4284,"STL":82686,
         "TOR":11141,"VAN":6321}
assert set(xwalk.values()) == set(mls_clubs.club_id)

salaries["club_id"] = salaries["Team"].map(xwalk)
salaries["Date"] = pd.to_datetime(salaries["Date"])
print("unmapped teams:", salaries.club_id.isna().sum())
print(salaries.dtypes[["Date", "club_id"]])

valuations_n = (valuations.merge(players[["player_id", "name_n"]], on="player_id")
              .dropna(subset=["current_club_id", "name_n"])
              .assign(current_club_id=lambda d: d.current_club_id.astype("int64"))
              .sort_values("date"))

asa = (salaries.dropna(subset=["club_id"]).sort_values("Date")
          .groupby(["Season", "name_n", "club_id"], as_index=False).last())
asa["club_id"] = asa.club_id.astype("int64")
asa["g"] = asa["Guaranteed Compensation_num"]

m = pd.merge_asof(
    asa.sort_values("Date"),
    vals_n[["name_n", "current_club_id", "date", "market_value_in_eur"]],
    left_on="Date", right_on="date",
    left_by=["name_n", "club_id"], right_by=["name_n", "current_club_id"],
    tolerance=pd.Timedelta(days=365), direction="backward")
m["has_val"] = m.market_value_in_eur.notna()
m["in_players"] = m.name_n.isin(players.name_n)

by_season = m.groupby("Season").agg(
    n=("name_n", "size"),
    in_players=("in_players", "mean"),
    has_val=("has_val", "mean"))
by_season["salary_weighted_val"] = m.groupby("Season").apply(
    lambda d: d.loc[d.has_val, "g"].sum() / d.g.sum())
print(by_season.round(2))

m["band"] = pd.cut(m.g, [0, 1e5, 2e5, 5e5, 1e6, 2e6, np.inf], right=False)
print(m.groupby("band", observed=True)[["in_players", "has_val"]].agg(["mean", "size"]).round(2))

print(m[(~m.in_players) & (m.Season == 2024)].sort_values("g", ascending=False)
        [["name_n", "club_id", "g"]].head(15).to_string())

unmapped teams: 0
Date       datetime64[us]
club_id             int64
dtype: object
           n  in_players  has_val  salary_weighted_val
Season                                                
2016     619        0.33     0.10                 0.04
2017     693        0.39     0.15                 0.10
2018     730        0.47     0.20                 0.17
2019     749        0.54     0.26                 0.24
2020     771        0.60     0.36                 0.38
2021     897        0.63     0.40                 0.43
2022     936        0.69     0.46                 0.53
2023     976        0.73     0.55                 0.64
2024     971        0.82     0.57                 0.65
2025    1017        0.91     0.67                 0.74
2026     911        0.92     0.69                 0.81
                       in_players       has_val      
                             mean  size    mean  size
band                                                 
[0.0, 100000.0)              0.46  2669

**Question this exploration answered:** Can Transfermarkt market values (Kaggle `transfermarkt-datasets`, competition `MLS1`) serve as a target or reference value for MLS players, 2016–2025?

**Finding:** no for this dataset.

- Valuations exist for MLS clubs from the late 2000s, but the history is sparse early on. Unique MLS-valued players by year ranged from 91 (2016) to 745 (2025), against 605 to 988 players per season in the ASA salary table.
- Time-aware match (ASA player-season-club vs. last valuation at the same club within 365 days before the ASA snapshot): coverage was **10% in 2016**, **36% in 2020**, **67% in 2025**.
- Transfer fees are not a substitute: of 3,151 transfers involving MLS clubs (2016–2025), 713 have a positive fee, 1,857 are `0.0` and 581 are null. The table has no loan or free-transfer flag. And the MLS has a salary cap and roster rules which distorts transfer fees talks, thus the use of salaries as target variable in this project.

**Caveats on this conclusion**

- The conclusion applies to this dataset. Other Transfermarkt sources (direct scraping) were not tested.
- The cause of low early coverage (data collection vs. MLS roster structure) was not tested.

**What continues:** salary analysis using ASA salaries, ASA performance tables, and player characteristics.

**Still open for that analysis**

- Age, height, foot, country of birth, national team.
- `players.csv` match rate by name is __% (`in_players` above); date of birth can be taken from there for matched players, and the rest needs another source.

In [44]:
for s in ["ruidiaz", "wagner", "nagbe", "zardes"]:
    print(s, players[players.name.str.lower().str.contains(s, na=False)][["name", "date_of_birth", "last_season"]].head(5).to_string(), "\n")

ruidiaz Empty DataFrame
Columns: [name, date_of_birth, last_season]
Index: [] 

wagner                    name date_of_birth  last_season
3409      Sandro Wagner    1987-11-29         2018
34361   Wagner Leonardo    1999-07-23         2025
34913     Robert Wagner    2003-07-14         2024
37570  Christian Wagner    2003-02-13         2021
37917      Lukas Wagner    2004-11-19         2022 

nagbe Empty DataFrame
Columns: [name, date_of_birth, last_season]
Index: [] 

zardes Empty DataFrame
Columns: [name, date_of_birth, last_season]
Index: [] 



In [46]:
w = salaries[salaries.Season.between(2016, 2025)]
u = (w.groupby("name_n")
       .agg(first_season=("Season", "min"), last_season=("Season", "max"),
            seasons=("Season", "nunique"), max_guaranteed=("Guaranteed Compensation_num", "max"),
            team=("Team", "last"))
       .reset_index())
u["in_kaggle"] = u.name_n.isin(players.name_n)

print("unique players:", len(u), "| in Kaggle:", u.in_kaggle.sum(), "| missing:", (~u.in_kaggle).sum())
print(u.groupby("first_season").in_kaggle.agg(["size", "mean"]).round(2))

dups = players.name_n.value_counts()
print("ambiguous matches:", u[u.name_n.map(dups).fillna(0) > 1].shape[0])

print(players[players.name_n.str.contains("ruidiaz", na=False)][["name", "date_of_birth", "last_season"]])

import os; os.makedirs("../../data/interim", exist_ok=True)
u[~u.in_kaggle].to_csv("../../data/interim/players_missing_bio.csv", index=False)

unique players: 2675 | in Kaggle: 1574 | missing: 1101
              size  mean
first_season            
2016           605  0.33
2017           236  0.48
2018           237  0.50
2019           214  0.58
2020           216  0.64
2021           229  0.61
2022           225  0.69
2023           232  0.73
2024           213  0.85
2025           268  0.87
ambiguous matches: 85
Empty DataFrame
Columns: [name, date_of_birth, last_season]
Index: []
